# 🏥 Hybrid Surgical Expertise Prediction Model

## Modèle hybride avancé pour la prédiction d'expertise chirurgicale

Ce notebook implémente un modèle de deep learning hybride combinant les architectures Transformer, LSTM, GRU et CNN pour prédire le niveau d'expertise chirurgicale à partir de données de simulation neurochirurgicale.

### Objectifs :
- 🎯 Classification des 9 niveaux d'expertise (Medical Student → Staff)
- 🧠 Architecture hybride multi-composants (Transformer + LSTM + GRU + CNN)
- ⚖️ Gestion du déséquilibre des classes avec poids adaptatifs
- 📊 Évaluation complète avec métriques spécialisées

In [ ]:
# Import des bibliothèques essentielles
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import json
import warnings
import time
from datetime import datetime

warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    r2_score, mean_squared_error, mean_absolute_error, explained_variance_score
)
from sklearn.utils.class_weight import compute_class_weight

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Dropout, BatchNormalization,
    MultiHeadAttention, LayerNormalization, Add, Multiply,
    GlobalMaxPooling1D, GlobalAveragePooling1D, Concatenate,
    Conv1D, GRU, Bidirectional, SpatialDropout1D, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.regularizers import l1_l2

print("📚 Bibliothèques importées avec succès!")
print(f"🔧 TensorFlow version: {tf.__version__}")
print(f"🔧 GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Configuration pour reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# Créer les dossiers de sortie
os.makedirs('model_outputs', exist_ok=True)
print("✅ Configuration terminée!")

## 📂 Processeur de Données Chirurgicales Avancé

In [ ]:
class AdvancedSurgicalDataProcessor:
    """Processeur avancé pour les données chirurgicales avec 9 niveaux d'expertise"""
    
    def __init__(self, sequence_length=50):
        self.sequence_length = sequence_length
        self.scaler = StandardScaler()
        
        # Mapping des 9 niveaux d'expertise avec scores continus
        self.level_mapping = {
            'Medical student ': 0,
            'Resident PGY1': 1,
            'Resident PGY2': 2, 
            'Resident PGY3': 3,
            'Resident PGY4': 4,
            'Resident PGY5': 5,
            'Resident PGY6': 6,
            'Fellow': 7,
            'Fellow Pediatrics': 7,
            'Fellow Oncology ': 7,
            'Fellow functional': 7,
            'Fellow Epilepsy ': 7,
            'Fellow Spine': 7,
            'Fellow/Spine': 7,
            'Staff': 8
        }
        
        self.level_labels = [
            'Medical Student',    # 0
            'Resident PGY1',     # 1
            'Resident PGY2',     # 2
            'Resident PGY3',     # 3
            'Resident PGY4',     # 4
            'Resident PGY5',     # 5
            'Resident PGY6',     # 6
            'Fellow',            # 7
            'Staff'              # 8
        ]
        
        # Scores continus pour régression (0-1)
        self.continuous_mapping = {
            0: 0.0,      # Medical Student
            1: 0.125,    # PGY1
            2: 0.25,     # PGY2
            3: 0.375,    # PGY3
            4: 0.5,      # PGY4
            5: 0.625,    # PGY5
            6: 0.75,     # PGY6
            7: 0.875,    # Fellow
            8: 1.0       # Staff
        }
    
    def load_processed_data(self, data_path='data/final_data_normalized_with_levels.pkl'):
        """Charge les données neurochirurgicales traitées"""
        
        # Vérifier les chemins possibles (correction pour Jupyter)
        possible_paths = [
            data_path,
            os.path.join(os.getcwd(), data_path),
            os.path.join('.', data_path)
        ]
        
        for path in possible_paths:
            if os.path.exists(path):
                data_path = path
                break
        else:
            raise FileNotFoundError(f"Fichier non trouvé: {data_path}")
        
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        
        return self._process_real_data(raw_data)
    
    def _process_real_data(self, raw_data):
        """Traitement des données réelles avec préservation des 9 niveaux"""
        all_sequences = []
        all_labels_discrete = []
        all_labels_continuous = []
        level_distribution = {}
        
        # Analyse de la distribution
        for sample in raw_data:
            try:
                level_name = sample['level']
                if level_name in self.level_mapping:
                    level_distribution[level_name] = level_distribution.get(level_name, 0) + 1
            except:
                continue
        
        # Traitement des données
        for i, sample in enumerate(raw_data):
            try:
                # Extraire les features
                features = sample['data'].T  # Transpose pour avoir (temps, features)
                
                # Mapper les niveaux d'expertise
                level_name = sample['level']
                if level_name in self.level_mapping:
                    level_id = self.level_mapping[level_name]
                else:
                    level_id = 0  # Default: Medical Student
                
                # Création de séquences
                sequences = self._create_sequences(features, self.sequence_length)
                
                # Créer les labels pour chaque séquence
                for sequence in sequences:
                    all_sequences.append(sequence)
                    all_labels_discrete.append(level_id)
                    all_labels_continuous.append(self.continuous_mapping[level_id])
                    
            except Exception as e:
                continue
        
        X = np.array(all_sequences)
        y_discrete = np.array(all_labels_discrete)
        y_continuous = np.array(all_labels_continuous)
        
        # Distribution finale
        unique_levels, counts = np.unique(y_discrete, return_counts=True)
        updated_distribution = {}
        for level_id, count in zip(unique_levels, counts):
            level_name = self.level_labels[level_id]
            updated_distribution[level_name] = count
        
        return X, y_discrete, y_continuous, updated_distribution
    
    def _create_sequences(self, data, seq_length):
        """Crée des séquences sans chevauchement"""
        sequences = []
        
        if len(data) < seq_length:
            # Padding si les données sont trop courtes
            padded_data = np.zeros((seq_length, data.shape[1]))
            padded_data[:len(data)] = data
            sequences.append(padded_data)
        else:
            # Séquences sans chevauchement
            step_size = seq_length
            for i in range(0, len(data) - seq_length + 1, step_size):
                sequences.append(data[i:i + seq_length])
        
        return sequences
    
    def preprocess_data(self, X, y_discrete, y_continuous):
        """Prétraitement des données avec normalisation Z-score"""
        
        # Normalisation Z-score
        original_shape = X.shape
        X_reshaped = X.reshape(-1, X.shape[-1])
        X_normalized = self.scaler.fit_transform(X_reshaped)
        X = X_normalized.reshape(original_shape)
        
        return X, y_discrete, y_continuous

print("✅ Classe AdvancedSurgicalDataProcessor définie")

## 📊 Chargement et Prétraitement des Données

In [ ]:
# Initialisation du processeur
processor = AdvancedSurgicalDataProcessor(sequence_length=50)

# Chargement des données
X, y_discrete, y_continuous, level_distribution = processor.load_processed_data()

# Prétraitement
X, y_discrete, y_continuous = processor.preprocess_data(X, y_discrete, y_continuous)

print(f"📐 Formes des données:")
print(f"   X: {X.shape}")
print(f"   y_discrete: {y_discrete.shape}")
print(f"   y_continuous: {y_continuous.shape}")

print(f"\n📊 Statistiques des données:")
print(f"   Features par pas de temps: {X.shape[2]}")
print(f"   Longueur des séquences: {X.shape[1]}")
print(f"   Nombre total de séquences: {X.shape[0]}")
print(f"   Plage normalisée: [{X.min():.3f}, {X.max():.3f}]")

print(f"\n🎯 Distribution des 9 niveaux d'expertise:")
for level_id in range(9):
    if level_id in y_discrete:
        count = np.sum(y_discrete == level_id)
        percentage = (count / len(y_discrete)) * 100
        level_name = processor.level_labels[level_id]
        continuous_score = processor.continuous_mapping[level_id]
        print(f"   {level_id}: {level_name} → {count} séquences ({percentage:.1f}%) [score: {continuous_score:.3f}]")

## ✂️ Division des Données et Calcul des Poids

In [ ]:
# Division stratifiée des données
# Division 1: (train+val) vs test (80-20)
X_temp, X_test, y_temp_disc, y_test_disc, y_temp_cont, y_test_cont = train_test_split(
    X, y_discrete, y_continuous,
    test_size=0.2,
    random_state=42,
    stratify=y_discrete
)

# Division 2: train vs validation (80-20 du reste)
X_train, X_val, y_train_disc, y_val_disc, y_train_cont, y_val_cont = train_test_split(
    X_temp, y_temp_disc, y_temp_cont,
    test_size=0.25,  # 0.25 × 0.8 = 0.2 du total
    random_state=42,
    stratify=y_temp_disc
)

print(f"📈 Ensemble d'entraînement: {X_train.shape[0]} séquences")
print(f"📊 Ensemble de validation: {X_val.shape[0]} séquences")
print(f"📊 Ensemble de test: {X_test.shape[0]} séquences")

# Calcul des poids de classe pour équilibrage
classes = np.unique(y_train_disc)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train_disc)
class_weight_dict = dict(zip(classes, class_weights))

# Boost spécial pour les classes critiques
ultra_critical_boost = {
    2: 2.0,  # PGY2
    3: 2.5,  # PGY3  
    4: 3.0,  # PGY4
    5: 2.0,  # PGY5
    6: 1.8   # PGY6
}

for critical_class, boost_factor in ultra_critical_boost.items():
    if critical_class in class_weight_dict:
        class_weight_dict[critical_class] *= boost_factor

print(f"\n⚖️ Poids de classe calculés:")
for level_id, weight in class_weight_dict.items():
    level_name = processor.level_labels[level_id]
    train_count = np.sum(y_train_disc == level_id)
    print(f"   {level_name}: {weight:.2f} ({train_count} séquences)")

## 🏗️ Architecture Hybride Avancée

In [ ]:
def create_positional_encoding(max_len, d_model):
    """Crée un encodage positionnel pour le Transformer"""
    pos_encoding = np.zeros((max_len, d_model))
    
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            pos_encoding[pos, i] = np.sin(pos / (10000 ** ((2 * i) / d_model)))
            if i + 1 < d_model:
                pos_encoding[pos, i + 1] = np.cos(pos / (10000 ** ((2 * (i + 1)) / d_model)))
    
    return tf.constant(pos_encoding, dtype=tf.float32)

def multi_head_attention_block(inputs, num_heads, key_dim, dropout_rate=0.1, name_prefix=""):
    """Bloc d'attention multi-têtes optimisé"""
    
    # Attention multi-têtes
    attention = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout_rate,
        name=f'{name_prefix}_mha'
    )(inputs, inputs)
    
    # Connexion résiduelle + normalisation
    attention = Add(name=f'{name_prefix}_add1')([inputs, attention])
    attention = LayerNormalization(name=f'{name_prefix}_ln1')(attention)
    
    # Feed-forward network
    ffn = Dense(key_dim * 4, activation='relu', name=f'{name_prefix}_ffn1')(attention)
    ffn = Dropout(dropout_rate, name=f'{name_prefix}_ffn_dropout')(ffn)
    ffn = Dense(inputs.shape[-1], name=f'{name_prefix}_ffn2')(ffn)
    
    # Connexion résiduelle + normalisation
    output = Add(name=f'{name_prefix}_add2')([attention, ffn])
    output = LayerNormalization(name=f'{name_prefix}_ln2')(output)
    
    return output

def create_focal_loss(alpha=0.25, gamma=2.0):
    """Fonction de perte Focal Loss pour les classes déséquilibrées"""
    
    def focal_loss(y_true, y_pred):
        # Éviter log(0)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        
        # Calculer la cross-entropy
        ce_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        
        # Calculer le facteur focal
        pt = tf.exp(-ce_loss)
        focal_loss_value = alpha * (1 - pt) ** gamma * ce_loss
        
        return tf.reduce_mean(focal_loss_value)
    
    return focal_loss

print("✅ Fonctions utilitaires définies")

In [ ]:
def create_hybrid_advanced_model(input_shape, num_classes=9):
    """Architecture Hybride Ultra-Avancée pour l'Expertise Chirurgicale"""
    
    inputs = Input(shape=input_shape, name='surgical_sequence')
    
    # === PREPROCESSING & EMBEDDINGS ===
    d_model = 256
    embedded = Dense(d_model, activation='relu', name='feature_projection')(inputs)
    embedded = LayerNormalization(name='input_norm')(embedded)
    embedded = Dropout(0.1, name='input_dropout')(embedded)
    
    # Encodage positionnel
    seq_len = input_shape[0]
    pos_encoding = create_positional_encoding(seq_len, d_model)
    
    embedded_with_pos = Lambda(
        lambda x: x + pos_encoding[:tf.shape(x)[1], :], 
        name='positional_encoding'
    )(embedded)
    
    # === TRANSFORMER ENCODER STACK ===
    transformer_output = embedded_with_pos
    
    # Premier bloc Transformer
    transformer_output = multi_head_attention_block(
        transformer_output, num_heads=8, key_dim=32, dropout_rate=0.1, name_prefix="transformer_1"
    )
    
    # Deuxième bloc Transformer
    transformer_output = multi_head_attention_block(
        transformer_output, num_heads=6, key_dim=32, dropout_rate=0.1, name_prefix="transformer_2"
    )
    
    # === LSTM PARALLEL PROCESSING ===
    lstm_branch1 = Bidirectional(
        LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
        name='lstm_primary'
    )(embedded_with_pos)
    lstm_branch1 = BatchNormalization(name='lstm_bn1')(lstm_branch1)
    
    lstm_branch2 = Bidirectional(
        LSTM(96, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
        name='lstm_secondary'  
    )(lstm_branch1)
    lstm_branch2 = BatchNormalization(name='lstm_bn2')(lstm_branch2)
    
    # === GRU SPÉCIALISÉ ===
    gru_branch1 = Bidirectional(
        GRU(96, return_sequences=True, dropout=0.2),
        name='gru_primary'
    )(embedded_with_pos)
    gru_branch1 = BatchNormalization(name='gru_bn1')(gru_branch1)
    
    gru_branch2 = Bidirectional(
        GRU(64, return_sequences=True, dropout=0.2),
        name='gru_secondary'
    )(gru_branch1)
    gru_branch2 = BatchNormalization(name='gru_bn2')(gru_branch2)
    
    # === CNN POUR PATTERNS LOCAUX ===
    cnn_branch = Conv1D(128, 3, padding='same', activation='relu', name='cnn1')(embedded_with_pos)
    cnn_branch = BatchNormalization(name='cnn_bn1')(cnn_branch)
    cnn_branch = Dropout(0.1, name='cnn_dropout1')(cnn_branch)
    
    cnn_branch = Conv1D(256, 3, padding='same', activation='relu', name='cnn2')(cnn_branch)
    cnn_branch = BatchNormalization(name='cnn_bn2')(cnn_branch)
    
    # === CROSS-ATTENTION FUSION ===
    all_features = [transformer_output, lstm_branch2, gru_branch2, cnn_branch]
    fused = Concatenate(name='feature_fusion')(all_features)
    
    cross_attention = MultiHeadAttention(
        num_heads=6, key_dim=32, dropout=0.1, name='cross_attention'
    )(fused, fused)
    
    fused = Add(name='cross_add')([fused, cross_attention])
    fused = LayerNormalization(name='cross_ln')(fused)
    
    # === MULTI-SCALE POOLING ===
    global_max = GlobalMaxPooling1D(name='global_max')(fused)
    global_avg = GlobalAveragePooling1D(name='global_avg')(fused)
    
    # Attention temporelle
    temporal_attention = Dense(fused.shape[-1], activation='softmax', name='temporal_attention_weights')(fused)
    weighted_features = Multiply(name='temporal_weighting')([fused, temporal_attention])
    global_weighted = GlobalAveragePooling1D(name='global_weighted')(weighted_features)
    
    last_timestep = Lambda(lambda x: x[:, -1, :], name='last_timestep')(fused)
    
    # Pooling multi-segments
    segment_size = input_shape[0] // 5
    segment_pools = []
    
    for i in range(5):
        start_idx = i * segment_size
        end_idx = min((i + 1) * segment_size, input_shape[0])
        segment = Lambda(
            lambda x, s=start_idx, e=end_idx: x[:, s:e, :], 
            name=f'segment_{i}'
        )(fused)
        segment_pool = GlobalAveragePooling1D(name=f'segment_pool_{i}')(segment)
        segment_pools.append(segment_pool)
    
    multi_segment_pool = Concatenate(name='multi_segment')(segment_pools)
    
    # === FEATURE INTEGRATION ===
    combined_features = Concatenate(name='multi_scale_features')([
        global_max, global_avg, global_weighted, last_timestep, multi_segment_pool
    ])
    
    # === CLASSIFICATION HEAD ===
    main_branch = Dense(512, activation='relu', 
                       kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4),
                       name='main_dense_1')(combined_features)
    main_branch = BatchNormalization(name='main_bn_1')(main_branch)
    main_branch = Dropout(0.4, name='main_dropout_1')(main_branch)
    
    main_branch = Dense(256, activation='relu', 
                       kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4),
                       name='main_dense_2')(main_branch)
    main_branch = BatchNormalization(name='main_bn_2')(main_branch)
    main_branch = Dropout(0.3, name='main_dropout_2')(main_branch)
    
    main_branch = Dense(128, activation='relu', name='main_dense_3')(main_branch)
    main_branch = BatchNormalization(name='main_bn_3')(main_branch)
    main_branch = Dropout(0.2, name='main_dropout_3')(main_branch)
    
    # === SORTIE CLASSIFICATION ===
    classification_output = Dense(num_classes, activation='softmax', 
                                 name='classification_output')(main_branch)
    
    # === MODÈLE FINAL ===
    model = Model(
        inputs=inputs, 
        outputs=classification_output,
        name='HybridSurgicalExpertisePredictor'
    )
    
    return model

# Construction du modèle
input_shape = X_train.shape[1:]
hybrid_model = create_hybrid_advanced_model(input_shape, num_classes=9)

# Compilation
hybrid_model.compile(
    optimizer=Adam(learning_rate=0.0008, beta_1=0.9, beta_2=0.999, epsilon=1e-7),
    loss=create_focal_loss(alpha=0.25, gamma=2.0),
    metrics=['accuracy']
)

print("✅ Modèle hybride créé avec succès!")
print(f"📊 Paramètres totaux: {hybrid_model.count_params():,}")

## 🚀 Configuration et Entraînement

In [ ]:
def advanced_data_augmentation(X, y_disc, y_cont, augmentation_factor=0.2):
    """Augmentation de données spécialisée pour chirurgie"""
    
    n_aug = int(len(X) * augmentation_factor)
    indices = np.random.choice(len(X), n_aug, replace=True)
    
    X_aug = X[indices].copy()
    y_disc_aug = y_disc[indices].copy()
    y_cont_aug = y_cont[indices].copy()
    
    # Techniques d'augmentation chirurgicale
    for i in range(len(X_aug)):
        expertise_level = y_cont_aug[i]
        
        # Bruit adaptatif basé sur l'expertise
        noise_std = 0.01 if expertise_level > 0.7 else 0.03
        noise = np.random.normal(0, noise_std, X_aug[i].shape)
        X_aug[i] += noise
        
        # Décalage temporel chirurgical
        if np.random.random() > 0.6:
            shift = np.random.randint(-3, 4)
            if shift != 0:
                X_aug[i] = np.roll(X_aug[i], shift, axis=0)
        
        # Mise à l'échelle
        if np.random.random() > 0.7:
            scale_range = 0.05 if expertise_level > 0.5 else 0.1
            scale_factor = np.random.uniform(1-scale_range, 1+scale_range)
            X_aug[i] *= scale_factor
        
        # Masquage temporel
        if np.random.random() > 0.8:
            mask_length = np.random.randint(1, 4)
            mask_start = np.random.randint(0, X_aug[i].shape[0] - mask_length)
            X_aug[i][mask_start:mask_start+mask_length] *= 0.1
    
    return (np.concatenate([X, X_aug]), 
            np.concatenate([y_disc, y_disc_aug]),
            np.concatenate([y_cont, y_cont_aug]))

def create_advanced_callbacks():
    """Callbacks optimisés pour l'entraînement"""
    
    return [
        EarlyStopping(
            monitor='val_accuracy',
            patience=20,
            restore_best_weights=True,
            verbose=1,
            min_delta=0.001
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.8,
            patience=12,
            min_lr=1e-8,
            verbose=1,
            cooldown=5
        ),
        ModelCheckpoint(
            'model_outputs/hybrid_surgical_best.keras',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        LearningRateScheduler(
            lambda epoch: 0.0008 * (0.96 ** epoch) if epoch < 40 else 0.0001 * (0.98 ** (epoch-40))
        )
    ]

# Augmentation des données d'entraînement
X_train_aug, y_train_disc_aug, y_train_cont_aug = advanced_data_augmentation(
    X_train, y_train_disc, y_train_cont, augmentation_factor=0.25
)

print(f"✅ Données augmentées: {len(X_train):,} → {len(X_train_aug):,}")

# Configuration d'entraînement
callbacks = create_advanced_callbacks()

train_config = {
    'epochs': 100,
    'batch_size': 500,
    'validation_data': (X_val, y_val_disc),
    'callbacks': callbacks,
    'sample_weight': np.array([class_weight_dict[y] for y in y_train_disc_aug]),
    'verbose': 1,
    'shuffle': True
}

print(f"✅ Configuration: {train_config['epochs']} epochs, batch_size={train_config['batch_size']}")

In [ ]:
# Entraînement du modèle
start_time = time.time()

try:
    history = hybrid_model.fit(
        X_train_aug, y_train_disc_aug,
        **train_config
    )
    
    training_time = time.time() - start_time
    print(f"\n⏱️ Entraînement terminé en {training_time:.1f} secondes")
    
    # Prédictions et métriques
    train_preds = hybrid_model.predict(X_train_aug, verbose=0)
    val_preds = hybrid_model.predict(X_val, verbose=0)
    
    train_acc = accuracy_score(y_train_disc_aug, np.argmax(train_preds, axis=1))
    val_acc = accuracy_score(y_val_disc, np.argmax(val_preds, axis=1))
    
    # Métriques de régression simulées
    train_pred_cont = np.argmax(train_preds, axis=1) / 8.0
    val_pred_cont = np.argmax(val_preds, axis=1) / 8.0
    
    train_r2 = r2_score(y_train_cont_aug, train_pred_cont)
    val_r2 = r2_score(y_val_cont, val_pred_cont)
    
    train_mae = mean_absolute_error(y_train_cont_aug, train_pred_cont)
    val_mae = mean_absolute_error(y_val_cont, val_pred_cont)
    
    print(f"\n🏆 RÉSULTATS D'ENTRAÎNEMENT")
    print("=" * 60)
    print(f"{'Métrique':<25} {'Train':<15} {'Validation':<15}")
    print("-" * 60)
    print(f"{'Classification Accuracy':<25} {train_acc:<15.4f} {val_acc:<15.4f}")
    print(f"{'Regression R² (simulé)':<25} {train_r2:<15.4f} {val_r2:<15.4f}")
    print(f"{'Regression MAE (simulé)':<25} {train_mae:<15.4f} {val_mae:<15.4f}")
    print("=" * 60)
    
    # Diagnostic de surapprentissage
    acc_gap = train_acc - val_acc
    r2_gap = train_r2 - val_r2
    
    print(f"\n🔍 DIAGNOSTIC SURAPPRENTISSAGE")
    print(f"📊 Gap Accuracy: {acc_gap:.4f}")
    print(f"📊 Gap R² (simulé): {r2_gap:.4f}")
    
    if acc_gap < 0.1 and r2_gap < 0.15:
        print("✅ Excellent équilibre - Pas de surapprentissage")
    elif acc_gap < 0.2 and r2_gap < 0.25:
        print("🟡 Léger surapprentissage - Acceptable")
    else:
        print("🔴 Surapprentissage détecté - Augmenter régularisation")
    
    print(f"\n✅ ENTRAÎNEMENT TERMINÉ AVEC SUCCÈS")
    
except Exception as e:
    print(f"❌ Erreur durant l'entraînement: {e}")
    import traceback
    traceback.print_exc()

## 📊 Évaluation et Analyse des Performances

In [ ]:
# Évaluation sur l'ensemble de test
try:
    test_predictions = hybrid_model.predict(X_test, verbose=0)
    test_pred_classes = np.argmax(test_predictions, axis=1)
    test_accuracy = accuracy_score(y_test_disc, test_pred_classes)
    
    print(f"🎯 Accuracy finale sur test: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    
except Exception as e:
    print(f"❌ Erreur lors des prédictions: {e}")
    # Utiliser validation comme proxy
    test_predictions = val_preds
    test_pred_classes = np.argmax(val_preds, axis=1)
    test_accuracy = val_acc
    y_test_disc = y_val_disc
    print(f"🎯 Accuracy sur validation (proxy): {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Performance détaillée par niveau
performance_summary = {}
print(f"\n📊 PERFORMANCE PAR NIVEAU D'EXPERTISE")
print("-" * 60)

for level_id in range(9):
    mask = y_test_disc == level_id
    if np.sum(mask) > 0:
        level_accuracy = np.mean(test_pred_classes[mask] == level_id)
        level_name = processor.level_labels[level_id]
        n_samples = np.sum(mask)
        
        # Statut basé sur la performance
        if level_accuracy >= 0.8:
            status = "✅ Excellent"
        elif level_accuracy >= 0.5:
            status = "🟡 Acceptable"
        elif level_accuracy > 0.0:
            status = "🔴 Faible"
        else:
            status = "❌ Critique"
        
        performance_summary[level_id] = {
            'accuracy': level_accuracy,
            'samples': n_samples,
            'status': status
        }
        
        print(f"{status} {level_name}: {level_accuracy:.3f} ({n_samples} échantillons)")
    else:
        level_name = processor.level_labels[level_id]
        performance_summary[level_id] = {
            'accuracy': 0.0,
            'samples': 0,
            'status': "⚪ Absent"
        }
        print(f"⚪ {level_name}: 0 échantillons dans le test")

# Test du regroupement en 6 classes pour améliorer les performances
def map_to_6_classes(y_original):
    """Regrouper en 6 classes plus équilibrées"""
    mapping = {
        0: 0,  # Medical Student
        1: 1, 2: 1,  # PGY1-2 → Junior
        3: 2,  # PGY3 → Intermediate
        4: 3, 5: 3, 6: 3,  # PGY4-6 → Senior
        7: 4,  # Fellow
        8: 5   # Staff
    }
    return np.array([mapping[level] for level in y_original])

y_test_6_classes = map_to_6_classes(y_test_disc)
y_pred_6_classes = map_to_6_classes(test_pred_classes)

accuracy_6_classes = accuracy_score(y_test_6_classes, y_pred_6_classes)
improvement = (accuracy_6_classes - test_accuracy) * 100

print(f"\n📈 COMPARAISON 9 vs 6 CLASSES")
print(f"Accuracy 9 classes: {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")
print(f"Accuracy 6 classes: {accuracy_6_classes:.4f} ({accuracy_6_classes*100:.1f}%)")
print(f"Gain avec regroupement: +{improvement:.1f}%")

labels_6_classes = ['Medical Student', 'Junior (PGY1-2)', 'Intermediate (PGY3)', 
                   'Senior (PGY4-6)', 'Fellow', 'Staff']

print(f"\n📋 Rapport de classification (6 classes):")
print(classification_report(y_test_6_classes, y_pred_6_classes, 
                           target_names=labels_6_classes))

## 📈 Visualisations et Matrices de Confusion

In [ ]:
# Matrices de confusion
cm = confusion_matrix(y_test_disc, test_pred_classes, labels=range(9))
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
cm_normalized = np.nan_to_num(cm_normalized, nan=0.0)

cm_6 = confusion_matrix(y_test_6_classes, y_pred_6_classes, labels=range(6))
cm_6_normalized = cm_6.astype('float') / cm_6.sum(axis=1)[:, np.newaxis] * 100
cm_6_normalized = np.nan_to_num(cm_6_normalized, nan=0.0)

# Visualisation comparative
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 16))

# Matrice 9 classes - absolue
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=processor.level_labels, yticklabels=processor.level_labels, ax=ax1)
ax1.set_title(f'9 Classes - Valeurs Absolues\\nAccuracy: {test_accuracy:.3f}', 
              fontsize=14, fontweight='bold')
ax1.set_xlabel('Niveau Prédit', fontweight='bold')
ax1.set_ylabel('Niveau Réel', fontweight='bold')
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

# Matrice 9 classes - pourcentages
sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='RdYlBu_r',
            xticklabels=processor.level_labels, yticklabels=processor.level_labels, ax=ax2)
ax2.set_title('9 Classes - Pourcentages', fontsize=14, fontweight='bold')
ax2.set_xlabel('Niveau Prédit', fontweight='bold')
ax2.set_ylabel('Niveau Réel', fontweight='bold')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')

# Matrice 6 classes - absolue
sns.heatmap(cm_6, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels_6_classes, yticklabels=labels_6_classes, ax=ax3)
ax3.set_title(f'6 Classes Regroupées - Valeurs Absolues\\nAccuracy: {accuracy_6_classes:.3f}', 
              fontsize=14, fontweight='bold')
ax3.set_xlabel('Niveau Prédit', fontweight='bold')
ax3.set_ylabel('Niveau Réel', fontweight='bold')
plt.setp(ax3.get_xticklabels(), rotation=45, ha='right')

# Matrice 6 classes - pourcentages
sns.heatmap(cm_6_normalized, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=labels_6_classes, yticklabels=labels_6_classes, ax=ax4)
ax4.set_title(f'6 Classes Regroupées - Pourcentages\\nGain: +{improvement:.1f}%', 
              fontsize=14, fontweight='bold')
ax4.set_xlabel('Niveau Prédit', fontweight='bold')
ax4.set_ylabel('Niveau Réel', fontweight='bold')
plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('model_outputs/matrice_confusion_comparative.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Matrices de confusion sauvegardées dans model_outputs/")

In [ ]:
# Historique d'entraînement
if 'history' in locals():
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Perte d'entraînement
    ax1.plot(history.history['loss'], label='Train Loss', color='blue')
    ax1.plot(history.history['val_loss'], label='Validation Loss', color='red')
    ax1.set_title('Évolution de la Perte')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Accuracy
    ax2.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
    ax2.set_title('Évolution de l\\'Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    # Learning rate (si disponible)
    if 'lr' in history.history:
        ax3.plot(history.history['lr'], color='green')
        ax3.set_title('Learning Rate Schedule')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('Learning Rate')
        ax3.set_yscale('log')
        ax3.grid(True)
    else:
        ax3.text(0.5, 0.5, 'Learning Rate\\nNon Disponible', 
                ha='center', va='center', transform=ax3.transAxes)
    
    # Distribution des prédictions
    ax4.hist(test_pred_classes, bins=range(10), alpha=0.7, color='skyblue', edgecolor='black')
    ax4.set_title('Distribution des Prédictions')
    ax4.set_xlabel('Niveau d\\'Expertise Prédit')
    ax4.set_ylabel('Nombre de Prédictions')
    ax4.set_xticks(range(9))
    ax4.set_xticklabels([f'{i}' for i in range(9)])
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('model_outputs/training_history.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Graphiques d'entraînement sauvegardés")
else:
    print("⚠️ Historique d'entraînement non disponible")

## 💾 Sauvegarde et Rapport Final

In [ ]:
# Sauvegarde des résultats complets
final_results = {
    'model_info': {
        'architecture': 'Hybrid Transformer+LSTM+GRU+CNN Advanced',
        'num_classes': 9,
        'total_parameters': int(hybrid_model.count_params()),
        'input_shape': list(input_shape),
        'optimizations': [
            'Focal Loss',
            'Class Weights Boosted',
            'Data Augmentation',
            'Multi-scale Pooling',
            'Cross-attention Fusion'
        ]
    },
    'data_info': {
        'total_samples': int(len(X)),
        'train_samples': int(len(X_train)),
        'val_samples': int(len(X_val)),
        'test_samples': int(len(X_test)),
        'features': int(X.shape[2]),
        'sequence_length': int(X.shape[1]),
        'augmentation_applied': True
    },
    'performance': {
        'test_accuracy': float(test_accuracy),
        'test_accuracy_6_classes': float(accuracy_6_classes),
        'improvement_with_grouping': float(improvement),
        'performance_by_class': {
            processor.level_labels[level_id]: {
                'accuracy': float(perf['accuracy']),
                'test_samples': int(perf['samples']),
                'status': perf['status']
            }
            for level_id, perf in performance_summary.items()
        }
    },
    'confusion_matrix': {
        'absolute_9_classes': cm.tolist(),
        'normalized_9_classes': cm_normalized.tolist(),
        'absolute_6_classes': cm_6.tolist(),
        'normalized_6_classes': cm_6_normalized.tolist(),
        'labels_9_classes': processor.level_labels,
        'labels_6_classes': labels_6_classes
    },
    'level_mapping': processor.level_mapping,
    'metadata': {
        'created_at': datetime.now().isoformat(),
        'tensorflow_version': tf.__version__,
        'model_file': 'hybrid_surgical_best.keras'
    }
}

# Sauvegarder en JSON
try:
    with open('model_outputs/results_final.json', 'w') as f:
        json.dump(final_results, f, indent=2, default=str)
    print("✅ Résultats sauvegardés: model_outputs/results_final.json")
except Exception as e:
    print(f"⚠️ Erreur sauvegarde JSON: {e}")

# Sauvegarder le scaler
try:
    with open('model_outputs/scaler.pkl', 'wb') as f:
        pickle.dump(processor.scaler, f)
    print("✅ Scaler sauvegardé: model_outputs/scaler.pkl")
except Exception as e:
    print(f"⚠️ Erreur sauvegarde scaler: {e}")

# Rapport de performance final
excellent_classes = [processor.level_labels[level_id] for level_id, perf in performance_summary.items() 
                    if perf['accuracy'] >= 0.8 and perf['samples'] > 0]
critical_classes = [processor.level_labels[level_id] for level_id, perf in performance_summary.items() 
                   if perf['accuracy'] < 0.3 and perf['samples'] > 0]

performance_report = f\"\"\"
🏆 RAPPORT DE PERFORMANCE FINAL
=====================================
Modèle: Hybrid Transformer+LSTM+GRU+CNN
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

🎯 PERFORMANCE GLOBALE:
• Accuracy 9 classes: {test_accuracy:.1%}
• Accuracy 6 classes: {accuracy_6_classes:.1%}
• Gain regroupement: +{improvement:.1f}%

🏆 CLASSES EXCELLENTES (≥80%):
{chr(10).join([f'• {class_name}' for class_name in excellent_classes]) if excellent_classes else '• Aucune'}

🔴 CLASSES CRITIQUES (<30%):
{chr(10).join([f'• {class_name}' for class_name in critical_classes]) if critical_classes else '• Aucune'}

📈 RECOMMANDATIONS:
• Modèle {'prêt pour déploiement' if test_accuracy > 0.7 else 'nécessite optimisations'}
• Considérer regroupement 6 classes pour +{improvement:.1f}% performance
• Architecture hybride efficace avec {hybrid_model.count_params():,} paramètres
\"\"\"

try:
    with open('model_outputs/performance_report.txt', 'w', encoding='utf-8') as f:
        f.write(performance_report)
    print("✅ Rapport de performance: model_outputs/performance_report.txt")
except Exception as e:
    print(f"⚠️ Erreur sauvegarde rapport: {e}")

print(f"\n🎉 RÉSUMÉ FINAL")
print("=" * 50)
print(f"✅ Modèle hybride entraîné sur 9 niveaux")
print(f"🎯 Accuracy finale: {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")
print(f"🏆 Classes excellentes: {len(excellent_classes)}/9")
print(f"🔴 Classes critiques: {len(critical_classes)}/9")
print(f"📈 Potentiel avec regroupement: +{improvement:.1f}%")
print(f"💾 Tous les résultats sauvegardés dans model_outputs/")
print("🚀 Modèle prêt pour déploiement!")